# Digital Advertising Budget Optimizer
## Notebook 01 — Environment Setup & Synthetic Data Generation

**Business question:** A company advertises on Facebook, Instagram, Google, TikTok and YouTube.
Management asks: *where should we spend the next Rs. 1,000,000?*

**Project roadmap**

| # | Notebook | Purpose |
|---|----------|---------|
| **01** | **Setup & data generation** | **Build the project on Google Drive and generate 10K → 10M row datasets** |
| 02 | EDA & KPIs (Pandas) | CTR, CPC, CPA, conversion rate, ROAS, campaign & audience performance |
| 03 | MapReduce (Hadoop Streaming) | Same KPIs as map/reduce jobs |
| 04 | Spark (PySpark) | Same KPIs with Spark DataFrames |
| 05 | Benchmark | Pandas vs MapReduce vs Spark across all data sizes |
| 06 | Recommendation engine | Response curves + optimiser → budget split for Rs. 1M |
| 07 | Final report / dashboard | Findings and recommendation for management |



In [1]:
# ============================================================
# 1. Environment & project folders
# ============================================================
import os, sys, shutil, time, json, platform
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_NAME = "ad-budget-optimizer"

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME   # permanent storage
    LOCAL_DIR = Path("/content") / PROJECT_NAME                  # fast scratch disk (wiped on restart)
else:                                                            # running locally / outside Colab
    DRIVE_DIR = LOCAL_DIR = Path.cwd() / PROJECT_NAME

FOLDERS = ["data/raw", "data/processed", "src", "config", "notebooks",
           "outputs/figures", "outputs/benchmarks", "outputs/reports"]
for f in FOLDERS:
    (DRIVE_DIR / f).mkdir(parents=True, exist_ok=True)
(LOCAL_DIR / "data/raw").mkdir(parents=True, exist_ok=True)

SRC_DIR = DRIVE_DIR / "src"
RAW_LOCAL = LOCAL_DIR / "data/raw"
RAW_DRIVE = DRIVE_DIR / "data/raw"
sys.path.insert(0, str(SRC_DIR))

# Machine specs (worth recording: benchmark results depend on them)
try:
    import psutil
    ram = f"{psutil.virtual_memory().total / 1e9:.1f} GB"
except ImportError:
    ram = "n/a"
print(f"Colab: {IN_COLAB} | Python {platform.python_version()} | CPUs: {os.cpu_count()} | RAM: {ram}")
print(f"Free local disk: {shutil.disk_usage(LOCAL_DIR).free / 1e9:.0f} GB")
print(f"Project folder : {DRIVE_DIR}")

Mounted at /content/drive
Colab: True | Python 3.13.18 | CPUs: 2 | RAM: 13.6 GB
Free local disk: 94 GB
Project folder : /content/drive/MyDrive/ad-budget-optimizer


## 2. Data design

**What "synthetic" means:** no company shares millions of rows of real ad data, so we *simulate* it.
The generator below encodes realistic market rules (each platform's cost, click rate, conversion rate,
order value, audience mix, seasonality, and diminishing returns on spend), then draws random records that follow them.

**Grain:** one row = one campaign's results for one audience segment (age × gender × city × device) on one day.
History covers 12 months: **1 Nov 2025 – 31 Oct 2026**.

| Column | Type | Description |
|---|---|---|
| date | date | Day of activity |
| campaign | string | e.g. `GG_Retargeting_01` (40 campaigns: 8 per platform) |
| platform | string | Facebook, Instagram, Google, TikTok, YouTube |
| impressions | int | Times the ad was shown |
| clicks | int | Clicks on the ad |
| spend | float | Cost in Rs. |
| conversions | int | Purchases / sign-ups |
| revenue | float | Revenue from conversions in Rs. |
| age | string | 18-24, 25-34, 35-44, 45-54, 55+ |
| gender | string | Male, Female |
| city | string | 10 Pakistani cities |
| device | string | Mobile, Desktop, Tablet |

**Built-in realism:** TikTok skews young, Google skews desktop; retargeting converts better than brand campaigns;
budgets spike around 11.11, Ramadan/Eid and 14 August; and every platform has **diminishing returns**
(doubling spend gives less than double revenue). That last rule is what the recommendation engine in Notebook 06 must discover.

The next cell saves the generator as `src/adgen.py` on Drive, so later notebooks can reuse it.

In [2]:
%%writefile {SRC_DIR}/adgen.py
"""
adgen.py — Synthetic digital-advertising data generator
Project: Digital Advertising Budget Optimizer (Big Data Analytics)

Grain: one row = one campaign's performance for one audience segment
(age x gender x city x device) on one day.

Every dataset size is produced by the SAME market rules below, so business
KPIs (CTR, CPC, CPA, ROAS) stay stable as the data grows; only volume changes.
"""
from __future__ import annotations

import json
import math
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq

# ----------------------------------------------------------------------------
# 1. Market rules (the hidden "ground truth" of the simulation)
# ----------------------------------------------------------------------------
SEED = 42
START_DATE = "2025-11-01"
END_DATE = "2026-10-31"

COLUMNS = ["date", "campaign", "platform", "impressions", "clicks", "spend",
           "conversions", "revenue", "age", "gender", "city", "device"]

# Baseline economics per platform at normal budget pressure.
#   cpm  = cost per 1,000 impressions (Rs)     ctr = clicks / impressions
#   cvr  = conversions / clicks                 aov = average order value (Rs)
#   beta = diminishing-returns exponent (revenue ~ spend^beta; 1.0 = no saturation)
#   hist_share = how the company has historically split its ad budget
PLATFORMS = {
    "Facebook":  {"code": "FB", "hist_share": 0.30, "cpm": 150, "ctr": 0.012, "cvr": 0.020, "aov": 2000, "beta": 0.80},
    "Instagram": {"code": "IG", "hist_share": 0.25, "cpm": 180, "ctr": 0.010, "cvr": 0.020, "aov": 3200, "beta": 0.75},
    "Google":    {"code": "GG", "hist_share": 0.20, "cpm": 900, "ctr": 0.030, "cvr": 0.040, "aov": 3400, "beta": 0.65},
    "TikTok":    {"code": "TT", "hist_share": 0.15, "cpm": 70,  "ctr": 0.008, "cvr": 0.010, "aov": 2300, "beta": 0.85},
    "YouTube":   {"code": "YT", "hist_share": 0.10, "cpm": 100, "ctr": 0.004, "cvr": 0.012, "aov": 3800, "beta": 0.80},
}

AGE_BANDS = ["18-24", "25-34", "35-44", "45-54", "55+"]
GENDERS = ["Male", "Female"]
CITIES = ["Karachi", "Lahore", "Islamabad", "Rawalpindi", "Faisalabad",
          "Multan", "Peshawar", "Hyderabad", "Quetta", "Sialkot"]
DEVICES = ["Mobile", "Desktop", "Tablet"]
CAMPAIGN_TYPES = ["Prospecting", "Retargeting", "Brand", "Promo"]
CAMPAIGNS_PER_TYPE = [3, 2, 1, 2]          # 8 campaigns per platform, 40 total

# Audience mix per platform (probabilities, same order as the lists above)
AGE_MIX = {
    "Facebook":  [0.22, 0.34, 0.22, 0.13, 0.09],
    "Instagram": [0.38, 0.37, 0.15, 0.07, 0.03],
    "Google":    [0.18, 0.32, 0.26, 0.15, 0.09],
    "TikTok":    [0.55, 0.30, 0.10, 0.04, 0.01],
    "YouTube":   [0.28, 0.33, 0.20, 0.12, 0.07],
}
GENDER_MIX = {
    "Facebook": [0.68, 0.32], "Instagram": [0.55, 0.45], "Google": [0.62, 0.38],
    "TikTok":   [0.60, 0.40], "YouTube":   [0.70, 0.30],
}
DEVICE_MIX = {
    "Facebook": [0.88, 0.09, 0.03], "Instagram": [0.95, 0.03, 0.02],
    "Google":   [0.70, 0.27, 0.03], "TikTok":    [0.98, 0.01, 0.01],
    "YouTube":  [0.80, 0.12, 0.08],
}
CITY_MIX = [0.24, 0.22, 0.10, 0.08, 0.08, 0.06, 0.07, 0.05, 0.04, 0.06]
CAMPAIGN_TYPE_MIX = [0.45, 0.20, 0.15, 0.20]

# Performance multipliers (1.0 = average)
AGE_CTR = [1.15, 1.05, 0.95, 0.90, 0.85]
AGE_CVR = [0.75, 1.10, 1.20, 1.05, 0.85]
GENDER_CVR = [0.95, 1.10]
CITY_CVR = [1.05, 1.08, 1.20, 1.00, 0.92, 0.88, 0.90, 0.85, 0.80, 0.95]
CITY_AOV = [1.10, 1.10, 1.25, 1.00, 0.90, 0.88, 0.90, 0.85, 0.85, 0.95]
DEVICE_CTR = [1.05, 0.85, 0.95]
DEVICE_CVR = [0.92, 1.45, 1.05]
CTYPE_CTR = [1.00, 1.30, 0.70, 1.15]
CTYPE_CVR = [0.80, 1.90, 0.45, 1.20]
WEEKDAY_CVR = [1.00, 1.00, 1.00, 1.00, 0.95, 1.05, 1.10]   # Mon..Sun (Friday Jummah dip)

# Seasonal budget pressure: the company spends more around these events
SEASONS = [
    ("2025-11-07", "2025-11-30", 1.40, "11.11 / Black Friday"),
    ("2025-12-01", "2025-12-31", 1.15, "Winter & wedding season"),
    ("2026-02-18", "2026-03-09", 1.30, "Ramadan"),
    ("2026-03-10", "2026-03-21", 1.60, "Eid ul-Fitr shopping"),
    ("2026-05-20", "2026-05-27", 1.25, "Eid ul-Adha"),
    ("2026-06-15", "2026-07-31", 0.85, "Summer lull"),
    ("2026-08-08", "2026-08-14", 1.30, "Independence Day sales"),
]

ROW_SPEND_MEDIAN = 1500     # Rs per row (segment-day)
ROW_SPEND_SIGMA = 0.70
DAILY_NOISE_SIGMA = 0.25    # random day-to-day budget swings per platform

NAMES = list(PLATFORMS)


# ----------------------------------------------------------------------------
# 2. Building blocks
# ----------------------------------------------------------------------------
def _noise(rng, sigma, n):
    """Log-normal noise with mean exactly 1."""
    return rng.lognormal(-sigma ** 2 / 2, sigma, n)


def build_market_calendar(seed=SEED):
    """Daily budget-pressure index per platform (mean 1.0).
    Identical for every dataset size, so all sizes share one market history."""
    dates = pd.date_range(START_DATE, END_DATE, freq="D")
    season = np.ones(len(dates))
    for start, end, mult, _ in SEASONS:
        season[(dates >= start) & (dates <= end)] = mult
    rng = np.random.default_rng(seed + 1000)
    pressure = np.empty((len(NAMES), len(dates)))
    for i in range(len(NAMES)):
        p = season * rng.lognormal(0, DAILY_NOISE_SIGMA, len(dates))
        pressure[i] = p / p.mean()
    return dates, pressure


def _normalisers():
    """Scale CTR and CVR so each platform's average CTR and ROAS match its baseline,
    regardless of its audience mix."""
    out = {}
    wk = np.mean(WEEKDAY_CVR)
    city_val = np.dot(CITY_MIX, np.multiply(CITY_CVR, CITY_AOV))
    for name in NAMES:
        a, d = np.array(AGE_MIX[name]), np.array(DEVICE_MIX[name])
        g, c = np.array(GENDER_MIX[name]), np.array(CAMPAIGN_TYPE_MIX)
        e_ctr = np.dot(a, AGE_CTR) * np.dot(d, DEVICE_CTR) * np.dot(c, CTYPE_CTR)
        e_val = (np.dot(a, np.multiply(AGE_CTR, AGE_CVR))
                 * np.dot(d, np.multiply(DEVICE_CTR, DEVICE_CVR))
                 * np.dot(c, np.multiply(CTYPE_CTR, CTYPE_CVR))
                 * np.dot(g, GENDER_CVR) * city_val * wk)
        out[name] = (1 / e_ctr, e_ctr / e_val)
    return out


def _campaign_names():
    names = []
    for name in NAMES:
        code = PLATFORMS[name]["code"]
        for ctype, k in zip(CAMPAIGN_TYPES, CAMPAIGNS_PER_TYPE):
            names += [f"{code}_{ctype}_{j + 1:02d}" for j in range(k)]
    return names


def _draw(rng, p_idx, mix_by_platform):
    """Vectorised categorical draw where probabilities depend on the platform."""
    cdf = np.cumsum(np.array([mix_by_platform[n] for n in NAMES]), axis=1)
    cdf[:, -1] = 1.0
    u = rng.random(len(p_idx))
    return (u[:, None] > cdf[p_idx]).sum(axis=1).astype(np.int8)


def _dict_col(codes, categories):
    return pa.DictionaryArray.from_arrays(pa.array(codes.astype(np.int8)), pa.array(categories))


# ----------------------------------------------------------------------------
# 3. Chunk generator
# ----------------------------------------------------------------------------
class _Market:
    def __init__(self, seed=SEED):
        self.dates, self.pressure = build_market_calendar(seed)
        n_days = len(self.dates)
        share = np.array([PLATFORMS[n]["hist_share"] for n in NAMES])
        probs = (share[:, None] * self.pressure / n_days).ravel()
        self.cell_probs = probs / probs.sum()
        self.n_days = n_days
        self.date_values = self.dates.values.astype("datetime64[D]")
        self.weekday = self.dates.dayofweek.values
        self.norms = _normalisers()
        self.campaigns = _campaign_names()
        self.p = {k: np.array([PLATFORMS[n][k] for n in NAMES], dtype=float)
                  for k in ["cpm", "ctr", "cvr", "aov", "beta"]}
        self.ctr_norm = np.array([self.norms[n][0] for n in NAMES])
        self.cvr_norm = np.array([self.norms[n][1] for n in NAMES])


def _generate_chunk(n, rng, mk):
    """Generate n rows as a pyarrow Table (categoricals dictionary-encoded)."""
    cell = rng.choice(len(mk.cell_probs), size=n, p=mk.cell_probs)
    p_idx, d_idx = cell // mk.n_days, cell % mk.n_days
    m = mk.pressure[p_idx, d_idx]

    age = _draw(rng, p_idx, AGE_MIX)
    gender = _draw(rng, p_idx, GENDER_MIX)
    device = _draw(rng, p_idx, DEVICE_MIX)
    city = _draw(rng, p_idx, {k: CITY_MIX for k in NAMES})
    ctype = _draw(rng, p_idx, {k: CAMPAIGN_TYPE_MIX for k in NAMES})

    per_type = np.array(CAMPAIGNS_PER_TYPE)
    offset = np.concatenate([[0], np.cumsum(per_type)[:-1]])
    k = (rng.random(n) * per_type[ctype]).astype(int)
    camp = p_idx * per_type.sum() + offset[ctype] + k

    spend = np.round(rng.lognormal(math.log(ROW_SPEND_MEDIAN), ROW_SPEND_SIGMA, n), 2)
    impressions = np.maximum(1, np.round(spend / mk.p["cpm"][p_idx] * 1000 * _noise(rng, 0.15, n))).astype(np.int64)

    ctr = (mk.p["ctr"][p_idx] * mk.ctr_norm[p_idx]
           * np.take(AGE_CTR, age) * np.take(DEVICE_CTR, device) * np.take(CTYPE_CTR, ctype)
           * _noise(rng, 0.20, n))
    clicks = rng.binomial(impressions, np.clip(ctr, 0, 0.5)).astype(np.int64)

    cvr = (mk.p["cvr"][p_idx] * mk.cvr_norm[p_idx]
           * np.take(AGE_CVR, age) * np.take(GENDER_CVR, gender) * np.take(CITY_CVR, city)
           * np.take(DEVICE_CVR, device) * np.take(CTYPE_CVR, ctype)
           * np.take(WEEKDAY_CVR, mk.weekday[d_idx])
           * m ** (mk.p["beta"][p_idx] - 1)            # diminishing returns
           * _noise(rng, 0.25, n))
    conversions = rng.binomial(clicks, np.clip(cvr, 0, 0.9)).astype(np.int64)
    revenue = np.round(conversions * mk.p["aov"][p_idx] * np.take(CITY_AOV, city) * _noise(rng, 0.30, n), 2)

    return pa.table({
        "date": pa.array(mk.date_values[d_idx], type=pa.date32()),
        "campaign": _dict_col(camp, mk.campaigns),
        "platform": _dict_col(p_idx, NAMES),
        "impressions": impressions,
        "clicks": clicks,
        "spend": spend,
        "conversions": conversions,
        "revenue": revenue,
        "age": _dict_col(age, AGE_BANDS),
        "gender": _dict_col(gender, GENDERS),
        "city": _dict_col(city, CITIES),
        "device": _dict_col(device, DEVICES),
    })


def _decode(table):
    """Turn dictionary columns into plain strings (needed for CSV)."""
    cols = [c.cast(c.type.value_type) if pa.types.is_dictionary(c.type) else c for c in table.columns]
    return pa.table(cols, names=table.column_names)


# ----------------------------------------------------------------------------
# 4. Public API
# ----------------------------------------------------------------------------
def preview(n=10, seed=SEED):
    """Small in-memory sample as a pandas DataFrame."""
    return _generate_chunk(n, np.random.default_rng(seed), _Market(seed)).to_pandas()


def _dir_mb(path):
    path = Path(path)
    files = [path] if path.is_file() else path.rglob("*")
    return sum(f.stat().st_size for f in files if f.is_file()) / 1e6


def generate_dataset(n_rows, out_dir, label, chunk_size=1_000_000, seed=SEED, overwrite=False):
    """Write n_rows to <out_dir>/csv/ads_<label>.csv and <out_dir>/parquet/ads_<label>/part-*.parquet.
    Generated in chunks, so RAM use stays flat no matter how big n_rows is."""
    out_dir = Path(out_dir)
    csv_path = out_dir / "csv" / f"ads_{label}.csv"
    pq_dir = out_dir / "parquet" / f"ads_{label}"
    if csv_path.exists() and pq_dir.exists() and not overwrite:
        print(f"[skip] {label}: already exists (set overwrite=True to rebuild)")
        return {"label": label, "rows": n_rows, "csv_mb": _dir_mb(csv_path),
                "parquet_mb": _dir_mb(pq_dir), "seconds": None}

    csv_path.parent.mkdir(parents=True, exist_ok=True)
    if pq_dir.exists():
        shutil.rmtree(pq_dir)
    pq_dir.mkdir(parents=True)

    mk = _Market(seed)
    n_chunks = math.ceil(n_rows / chunk_size)
    seeds = np.random.SeedSequence([seed, n_rows]).spawn(n_chunks)
    opts = pacsv.WriteOptions(include_header=False, quoting_style="none")
    t0, writer = time.time(), None
    sink = open(csv_path, "wb")
    sink.write((",".join(COLUMNS) + "\n").encode())          # clean, unquoted header
    try:
        for i, ss in enumerate(seeds):
            n = min(chunk_size, n_rows - i * chunk_size)
            table = _generate_chunk(n, np.random.default_rng(ss), mk)
            pq.write_table(table, pq_dir / f"part-{i:05d}.parquet", compression="snappy")
            plain = _decode(table)
            if writer is None:
                writer = pacsv.CSVWriter(sink, plain.schema, write_options=opts)
            writer.write_table(plain)
            print(f"  {label}: chunk {i + 1}/{n_chunks} ({(i * chunk_size + n):,} rows)", end="\r")
    finally:
        if writer is not None:
            writer.close()
        sink.close()
    secs = time.time() - t0
    print(f"[done] {label}: {n_rows:,} rows in {secs:.1f}s" + " " * 20)
    return {"label": label, "rows": n_rows, "csv_mb": _dir_mb(csv_path),
            "parquet_mb": _dir_mb(pq_dir), "seconds": round(secs, 1)}


def export_config(path, seed=SEED):
    """Save the market rules (answer key) as JSON, used later to validate the engine."""
    platforms = {}
    for name, p in PLATFORMS.items():
        roas = 1000 / p["cpm"] * p["ctr"] * p["cvr"] * p["aov"]
        platforms[name] = {**p, "baseline_cpc": round(p["cpm"] / 1000 / p["ctr"], 2),
                           "baseline_cpa": round(p["cpm"] / 1000 / p["ctr"] / p["cvr"], 2),
                           "baseline_roas": round(roas, 3)}
    cfg = {
        "seed": seed, "date_range": [START_DATE, END_DATE], "grain": "campaign x segment x day",
        "platforms": platforms,
        "seasons": [{"start": s, "end": e, "budget_multiplier": m, "event": n} for s, e, m, n in SEASONS],
        "row_spend": {"median_rs": ROW_SPEND_MEDIAN, "lognormal_sigma": ROW_SPEND_SIGMA},
        "daily_noise_sigma": DAILY_NOISE_SIGMA,
    }
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(cfg, indent=2))
    return cfg


def quick_kpis(parquet_dir):
    """Platform-level KPIs straight from Parquet (used for sanity checks)."""
    cols = ["platform", "impressions", "clicks", "spend", "conversions", "revenue"]
    df = pq.read_table(parquet_dir, columns=cols).to_pandas()
    g = df.groupby("platform", observed=True)[cols[1:]].sum()
    return pd.DataFrame({
        "spend_share_%": 100 * g.spend / g.spend.sum(),
        "CTR_%": 100 * g.clicks / g.impressions,
        "CPC_Rs": g.spend / g.clicks,
        "CVR_%": 100 * g.conversions / g.clicks,
        "CPA_Rs": g.spend / g.conversions,
        "ROAS": g.revenue / g.spend,
    }).round(2)

Writing /content/drive/MyDrive/ad-budget-optimizer/src/adgen.py


In [3]:
# ============================================================
# 3. Load generator, save the answer key, preview sample rows
# ============================================================
import importlib
import pandas as pd
import adgen
importlib.reload(adgen)

# The market rules as JSON = the "answer key".
# Notebook 06 will check whether the engine recovers these numbers from the data alone.
cfg = adgen.export_config(DRIVE_DIR / "config" / "generator_config.json")
print("Baseline ROAS by platform (ground truth):",
      {k: v["baseline_roas"] for k, v in cfg["platforms"].items()})

pd.set_option("display.width", 200)
adgen.preview(10)

Baseline ROAS by platform (ground truth): {'Facebook': 3.2, 'Instagram': 3.556, 'Google': 4.533, 'TikTok': 2.629, 'YouTube': 1.824}


,date,campaign,platform,impressions,clicks,spend,conversions,revenue,age,gender,city,device
0,2025-12-22,TT_Prospecting_03,TikTok,42432,349,2734.77,0,0.00,18-24,Female,Lahore,Mobile
1,2026-05-08,IG_Prospecting_02,Instagram,6881,39,1312.00,0,0.00,45-54,Male,Peshawar,Desktop
2,2026-07-19,TT_Brand_01,TikTok,9291,46,614.15,0,0.00,25-34,Female,Faisalabad,Mobile
3,2026-07-19,GG_Brand_01,Google,675,12,678.52,1,3409.26,45-54,Female,Lahore,Mobile
4,2026-02-19,FB_Brand_01,Facebook,4920,39,788.08,0,0.00,25-34,Female,Peshawar,Mobile
5,2026-08-01,YT_Brand_01,YouTube,19837,54,2124.37,0,0.00,18-24,Male,Peshawar,Mobile
6,2025-11-24,TT_Retargeting_02,TikTok,19566,119,1657.26,1,2090.34,25-34,Male,Lahore,Mobile
7,2026-01-21,TT_Retargeting_01,TikTok,36961,305,2432.22,2,4759.92,18-24,Male,Lahore,Mobile
8,2026-03-23,FB_Prospecting_01,Facebook,6834,44,1112.25,0,0.00,45-54,Male,Faisalabad,Mobile
9,2026-05-26,IG_Prospecting_02,Instagram,9225,73,1676.06,1,3499.23,25-34,Female,Karachi,Mobile


## 4. Generate the datasets

The **same generator** runs at four sizes. The rules never change, only the number of records,
so business answers (CTR, ROAS…) stay the same while processing load grows 1,000×.
That isolates the question we want to test: *how do Pandas, MapReduce and Spark cope as volume grows?*

Each size is saved in two formats:
- **CSV**: the format all three tools can read (required for a fair benchmark, and for Hadoop Streaming)
- **Parquet**: compressed, columnar big-data format (typically ~5× smaller and much faster to read)

Data is written to Colab's fast local disk first, then copied to Drive (writing millions of rows straight to Drive is slow).
Generation is chunked (1M rows at a time), so RAM stays flat even at 10M rows.

In [4]:
# ============================================================
# 4. Generate 10K -> 100K -> 1M -> 10M
# ============================================================
SCALES = {"10k": 10_000, "100k": 100_000, "1m": 1_000_000, "10m": 10_000_000}

# Optional stress tier: 50M rows (~4.4 GB CSV). Useful to show where Pandas runs out of RAM.
INCLUDE_STRESS_TIER = False
if INCLUDE_STRESS_TIER:
    SCALES["50m"] = 50_000_000

OVERWRITE = False   # True = regenerate even if files already exist

results = [adgen.generate_dataset(n, RAW_LOCAL, label, overwrite=OVERWRITE)
           for label, n in SCALES.items()]

summary = pd.DataFrame(results).set_index("label")
summary["csv_mb"] = summary["csv_mb"].round(1)
summary["parquet_mb"] = summary["parquet_mb"].round(1)
summary["compression_x"] = (summary["csv_mb"] / summary["parquet_mb"]).round(1)
summary.to_csv(DRIVE_DIR / "outputs/reports/01_dataset_summary.csv")
summary

[done] 10k: 10,000 rows in 0.0s                    
[done] 100k: 100,000 rows in 0.3s                    
[done] 1m: 1,000,000 rows in 2.2s                    
[done] 10m: 10,000,000 rows in 24.1s                    


,rows,csv_mb,parquet_mb,seconds,compression_x
label,,,,,
10k,10000,0.9,0.2,0.0,4.5
100k,100000,8.7,2.1,0.3,4.1
1m,1000000,87.3,18.0,2.2,4.8
10m,10000000,872.5,180.1,24.1,4.8


## 5. Validate: KPIs should be stable across sizes

If the generator is correct, every dataset should tell the same business story.
Small datasets will wobble a little (sampling noise); large ones should sit very close to the ground-truth baseline.

In [5]:
# ============================================================
# 5. KPI stability check across dataset sizes
# ============================================================
kpis = {label: adgen.quick_kpis(RAW_LOCAL / "parquet" / f"ads_{label}") for label in SCALES}

roas = pd.DataFrame({label: k["ROAS"] for label, k in kpis.items()})
roas["ground_truth"] = pd.Series({k: v["baseline_roas"] for k, v in cfg["platforms"].items()})
print("ROAS by platform and dataset size")
display(roas.round(2))

largest = list(SCALES)[-1]
print(f"\nFull KPI table ({largest} rows dataset)")
display(kpis[largest])

roas.to_csv(DRIVE_DIR / "outputs/reports/01_kpi_validation_roas.csv")

ROAS by platform and dataset size


,10k,100k,1m,10m,ground_truth
platform,,,,,
Facebook,3.25,3.20,3.19,3.18,3.20
Instagram,3.59,3.46,3.52,3.53,3.56
Google,4.30,4.44,4.48,4.49,4.53
TikTok,2.56,2.61,2.63,2.61,2.63
YouTube,1.70,1.80,1.82,1.81,1.82



Full KPI table (10m rows dataset)


,spend_share_%,CTR_%,CPC_Rs,CVR_%,CPA_Rs,ROAS
platform,,,,,,
Facebook,30.00,1.2,12.51,1.90,657.34,3.18
Instagram,24.99,1.0,18.00,1.90,946.97,3.53
Google,20.01,3.0,30.01,3.79,791.16,4.49
TikTok,15.00,0.8,8.75,0.95,918.90,2.61
YouTube,10.00,0.4,25.00,1.14,2196.01,1.81


## 6. Save datasets to Google Drive

Copies everything from local disk to `MyDrive/ad-budget-optimizer/data/raw/`.
Total is roughly 1.2 GB (about 6 GB with the stress tier). Drive syncs in the background,
so give it a few minutes before closing the browser tab.

In [6]:
# ============================================================
# 6. Copy local -> Drive
# ============================================================
if IN_COLAB:
    t0 = time.time()
    shutil.copytree(RAW_LOCAL, RAW_DRIVE, dirs_exist_ok=True)
    print(f"Copied to Drive in {time.time() - t0:.0f}s")

def folder_mb(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 1e6

print(f"\nContents of {RAW_DRIVE}:")
for p in sorted((RAW_DRIVE / "csv").glob("*.csv")):
    print(f"  csv/{p.name:<16} {p.stat().st_size / 1e6:>9.1f} MB")
for p in sorted((RAW_DRIVE / "parquet").iterdir()):
    print(f"  parquet/{p.name:<12} {folder_mb(p):>9.1f} MB  ({len(list(p.glob('*.parquet')))} files)")
print(f"\nTotal: {folder_mb(RAW_DRIVE) / 1000:.2f} GB")

Copied to Drive in 6s

Contents of /content/drive/MyDrive/ad-budget-optimizer/data/raw:
  csv/ads_100k.csv           8.7 MB
  csv/ads_10k.csv            0.9 MB
  csv/ads_10m.csv          872.5 MB
  csv/ads_1m.csv            87.3 MB
  parquet/ads_100k           2.1 MB  (1 files)
  parquet/ads_10k            0.2 MB  (1 files)
  parquet/ads_10m          180.1 MB  (10 files)
  parquet/ads_1m            18.0 MB  (1 files)

Total: 1.17 GB


## ✅ Notebook 01 complete

**Drive now contains**
```
ad-budget-optimizer/
├── config/generator_config.json      ← ground truth (answer key)
├── data/raw/csv/ads_{10k,100k,1m,10m}.csv
├── data/raw/parquet/ads_{...}/part-*.parquet
├── outputs/reports/01_*.csv          ← dataset summary & validation
└── src/adgen.py                      ← reusable generator
```

**Next: Notebook 02, EDA & KPIs with Pandas.** Later notebooks start by copying the data they need from Drive
to local disk, so benchmarks measure processing speed rather than Drive network speed.